# 04 — Selection verdict

Applies the pre-registered decision rule (`README.md`) to the notebook 01–03 results
on the even half, freezes ONE configuration per channel, and only then evaluates that
single configuration on the **odd half** (the confirmation — these numbers were not
inspected before the choice).

In [ ]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

sys.path.insert(0, os.getcwd())
import abcd_tools as at
import study_setup as ss

hep.style.use("CMS")
plt.rcParams["figure.figsize"] = (7, 5)
LUMI_LABEL = r"59.8 fb$^{-1}$ (13 TeV, 2018 sim.)"

def cms_label(ax=None):
    hep.cms.label("Work in progress", data=False, rlabel=LUMI_LABEL, ax=ax)

# pre-skim / pre-filter sums of gen weights (see README "Normalization")
SUMW_PRE = {}
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_2MU2E))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_4MU))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_BKG_UNSKIMMED))

# load everything, normalized (TTJets kept at the campaign 471.7 pb; the NNLO
# alternative is an explicit, documented rescale -- see the xsec table below)
bkg = {s: ss.load_normalized(s, SUMW_PRE)[0] for s in ss.BACKGROUNDS}
sig = {s: ss.load_normalized(s, SUMW_PRE)[0] for s in ss.SIGNALS_2MU2E + ss.SIGNALS_4MU}
by_process = {p: ss.sum_process(bkg, [s for s, pp in ss.BACKGROUNDS.items() if pp == p])
              for p in ss.PROCESSES}
total_bkg = ss.sum_process(bkg, list(ss.BACKGROUNDS))
print(f"loaded {len(bkg)} background + {len(sig)} signal samples")

In [ ]:
gates = json.load(open(os.path.join(ss.WORKDIR, "gates_even.json")))
sens = json.load(open(os.path.join(ss.WORKDIR, "sensitivity_even.json")))
import collections
verdict = {}
for ch in ss.PLANES:
    rows = []
    for pname in ss.PLANES[ch]:
        key = f"{ch}/{pname}"
        g1 = gates["gate1"].get(f"{key}/i", {})
        pass1 = all(np.isnan(v) or v > 0.05 for v in g1.values()) and any(np.isfinite(v) for v in g1.values())
        g2 = gates["gate2"][key]
        pass2 = min(g2["n_eff"].values()) >= 10
        nc = gates["gate3_nonclosure"].get(key)
        screened = pname in ss.SCREENED_ONLY[ch]
        pass3 = None if screened else (nc is not None and
                 all(v is not None and v <= max(0.25, 2 * g2["R_err"]) for v in nc.values() if v is not None))
        zs = [v["Z"] for k, v in sens.items() if k.startswith(key + "/")]
        rows.append((pname, pass1, pass2, pass3, screened, np.median(zs) if zs else np.nan,
                     min(g2["n_eff"].values())))
    print(f"=== {ch}")
    print(f"{'plane':20s} {'G1':>5s} {'G2':>5s} {'G3':>7s} {'medZ':>7s} {'min_neff':>9s}")
    for r in rows:
        g3s = "screen" if r[4] else str(r[3])
        print(f"{r[0]:20s} {str(r[1]):>5s} {str(r[2]):>5s} {g3s:>7s} {r[5]:7.3f} {r[6]:9.1f}")
    # screened-only planes (pass3 None) cannot be selected, per the decision rule
    surv = [r for r in rows if r[1] and r[2] and r[3]]
    if surv:
        best = max(surv, key=lambda r: (r[5], r[6]))
        verdict[ch] = best[0]
        print(f"--> chosen: {best[0]} (rank: median Z, tiebreak min n_eff)")
    else:
        verdict[ch] = None
        print("--> NO SURVIVOR (see extended-ABCD promotion trigger in README)")

## Plateau check and odd-half confirmation

The chosen working point must give the same verdict under ±1-bin boundary shifts;
then the frozen configuration is evaluated once on the odd half.

In [ ]:
for ch, pname in verdict.items():
    if pname is None:
        continue
    spec = ss.PLANES[ch][pname]
    # plateau: +-1 bin on each boundary
    vals, var, xe, ye = ss.plane_arrays(total_bkg, ch, pname, parity=0)
    ix = at.edge_index(xe, spec["xspec"][1]); iy = at.edge_index(ye, spec["yspec"][1])
    print(f"--- {ch}/{pname} plateau (even):")
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            reg = at.region_sums(vals, var, xe, ye,
                                 (spec["xspec"][0], float(xe[ix + dx])),
                                 (spec["yspec"][0], float(ye[iy + dy])))
            r, vr = at.closure_ratio(reg)
            print(f"  ({dx:+d},{dy:+d}): R = {r:6.3f} +- {np.sqrt(max(vr,0)):5.3f}")
    # the one look at the odd half
    ovals, ovar, oxe, oye = ss.plane_arrays(total_bkg, ch, pname, parity=1)
    oreg = at.region_sums(ovals, ovar, oxe, oye, spec["xspec"], spec["yspec"])
    orr, ovr = at.closure_ratio(oreg)
    print(f"  ODD-HALF confirmation: R = {orr:6.3f} +- {np.sqrt(max(ovr,0)):5.3f}, "
          f"A = {oreg['A'][0]:9.4g}")

## Final recommendation

*(written after execution: the chosen plane + WP + event-cut menu per channel, the
non-closure systematic to carry, the designated round-2 data validation region, and
the caveats — TTJets xsec sign-off, cosmic veto deferred to the data round.)*